Fundamentals of Deep Learning Models

# Lab 05-1: Convolution Operations for CNNs
## Exercise: Implementing core CNN functions with NumPy

This exercise implements the fundamental operations of **convolutional neural networks** (Sections 5.1–5.7):
the im2col/col2im data rearrangement (Section 5.7), 2D convolution forward and backward propagation
(Section 5.3, Eqs. 5.4–5.8), and pooling forward and backward propagation (Section 5.4, Eqs. 5.11–5.14).
All implementations use NumPy only, without deep learning frameworks.

### Load library packages

In [1]:
import numpy as np

print('NumPy version:', np.__version__)

NumPy version: 2.0.2


### Notations for convolutional neural networks

The following notation is used throughout this exercise, consistent with the textbook (Section 5.1):

| Symbol | Meaning | Textbook symbol |
|--------|---------|-----------------|
| $b$ | Batch size (number of samples) | $B$ |
| $x_h, x_w$ | Height and width of an input feature map | $N_H, N_W$ |
| $x_c$ | Number of input channels | $C_I$ |
| $f_h, f_w$ | Filter height and width | $F_H, F_W$ |
| $f_o$ | Number of output channels (number of filters) | $C_O$ |
| $f_i$ | Number of input channels of a filter (equals $x_c$) | $C_I$ |
| $s_h, s_w$ | Stride in height and width directions | $S$ |
| $p_h, p_w$ | Padding size in height and width directions | $P$ |

**Superscripts and subscripts**:
- Superscript $[l]$ denotes the $l$-th layer (e.g., $a^{[4]}$ is the 4th layer activation).
- Superscript $(i)$ denotes the $i$-th training example.
- The output spatial size after convolution or pooling follows Eq. (5.9):
  $N_O = \lfloor (N_I + 2P - F) / S \rfloor + 1$

### Dimensions of variables

#### Input image
- A single input example $\mathbf{x}^{(i)}$ is a 3D tensor of shape $(x_h, x_w, x_c)$.
- A mini-batch of $b$ examples is a 4D tensor of shape $(b, x_h, x_w, x_c)$.

#### Filter weights
- Each layer has $f_o$ filters. Each filter has shape $(f_h, f_w, f_i)$ where $f_i = x_c$.
- The full weight tensor of layer $[l]$ has shape $(f_o, f_h, f_w, f_i)$.

#### Output feature map
- The 4D output tensor has shape $(b, x_h^{[l]}, x_w^{[l]}, f_o)$,
  where $x_h^{[l]}$ and $x_w^{[l]}$ are determined by stride and padding via Eq. (5.9).

### Prepare convolution: im2col and col2im

The `im2col` function rearranges the 4D input tensor $(b, x_h, x_w, x_c)$ into a 2D matrix
of shape $(b \times n_h \times n_w,\; f_h \times f_w \times x_c)$, where each row is a flattened
local patch corresponding to one output spatial position (Section 5.7, Fig. 5.18).

This allows the convolution to be computed as a matrix multiplication with the reshaped
weight matrix of shape $(f_o,\; f_h \times f_w \times f_i)$.

The `col2im` function restores the 2D matrix back to the original 4D tensor shape.

With padding $P$ and stride $S$, the output feature map size is (Eq. 5.9):
$$N_O = \lfloor (N_I + 2P - F) / S \rfloor + 1$$

In [ ]:
def im2col(x_img, flt_h, flt_w, stride_h=1, stride_w=1, padding_h=1, padding_w=1):
    x_b, x_h, x_w, x_c = x_img.shape
    p_h = padding_h
    p_w = padding_w

    ### START CODE HERE ###

    # Zero-pad the input along spatial dimensions (Section 5.2)
    p_x = None
    # Compute output spatial dimensions (Eq. 5.9)
    n_h = None
    n_w = None
    # Allocate 6D buffer for extracted patches
    t_col = None

    # Extract each local patch from the padded input
    for i in range(n_h):
        h = i * stride_h
        h_l = h + flt_h
        for j in range(n_w):
            w = j * stride_w
            w_l = w + flt_w
            t_col[:, i, j, :, :, :] = None

    # Reshape to 2D: (b*n_h*n_w, f_h*f_w*x_c) (Section 5.7, Fig. 5.18)
    x_col = None

    ### END CODE HERE ###

    return x_col

def col2im(x_col, x_shape, flt_h=1, flt_w=1, stride_h=1, stride_w=1, padding_h=1, padding_w=1):
    x_b, x_h, x_w, x_c = x_shape
    p_h = padding_h
    p_w = padding_w

    ### START CODE HERE ###

    # Compute output spatial dimensions (Eq. 5.9)
    n_h = None
    n_w = None
    # Reshape 2D column matrix back to 6D patch buffer
    r_col = None
    # Allocate padded image buffer
    p_img = None

    # Place patches back into spatial positions
    for i in range(0, n_h, flt_h):
        h = i * stride_h
        h_l = h + flt_h
        for j in range(0, n_w, flt_w):
            w = j * stride_w
            w_l = w + flt_w
            p_img[:, h:h_l, w:w_l, :] = None

    # Remove padding to restore original spatial size
    x_img = None

    ### END CODE HERE ###

    return x_img

#### Test im2col and col2im

In [3]:
m_img = np.arange(1,17).reshape(1,4,4,1)
print(m_img.transpose(0,3,1,2))
c_img = im2col(m_img, 3, 3, 1, 1, 1, 1)
print(c_img)
b_img = col2im(c_img, m_img.shape, 3, 3, 1, 1, 1, 1)
print(b_img.transpose(0,3,1,2))

[[[[ 1  2  3  4]
   [ 5  6  7  8]
   [ 9 10 11 12]
   [13 14 15 16]]]]
[[ 0.  0.  0.  0.  1.  2.  0.  5.  6.]
 [ 0.  0.  0.  1.  2.  3.  5.  6.  7.]
 [ 0.  0.  0.  2.  3.  4.  6.  7.  8.]
 [ 0.  0.  0.  3.  4.  0.  7.  8.  0.]
 [ 0.  1.  2.  0.  5.  6.  0.  9. 10.]
 [ 1.  2.  3.  5.  6.  7.  9. 10. 11.]
 [ 2.  3.  4.  6.  7.  8. 10. 11. 12.]
 [ 3.  4.  0.  7.  8.  0. 11. 12.  0.]
 [ 0.  5.  6.  0.  9. 10.  0. 13. 14.]
 [ 5.  6.  7.  9. 10. 11. 13. 14. 15.]
 [ 6.  7.  8. 10. 11. 12. 14. 15. 16.]
 [ 7.  8.  0. 11. 12.  0. 15. 16.  0.]
 [ 0.  9. 10.  0. 13. 14.  0.  0.  0.]
 [ 9. 10. 11. 13. 14. 15.  0.  0.  0.]
 [10. 11. 12. 14. 15. 16.  0.  0.  0.]
 [11. 12.  0. 15. 16.  0.  0.  0.  0.]]
[[[[ 1.  2.  3.  4.]
   [ 5.  6.  7.  8.]
   [ 9. 10. 11. 12.]
   [13. 14. 15. 16.]]]]


**Expected Output:**
```
[[[[ 1  2  3  4]
   [ 5  6  7  8]
   [ 9 10 11 12]
   [13 14 15 16]]]]
[[ 0.  0.  0.  0.  1.  2.  0.  5.  6.]
 [ 0.  0.  0.  1.  2.  3.  5.  6.  7.]
 [ 0.  0.  0.  2.  3.  4.  6.  7.  8.]
 [ 0.  0.  0.  3.  4.  0.  7.  8.  0.]
 [ 0.  1.  2.  0.  5.  6.  0.  9. 10.]
 [ 1.  2.  3.  5.  6.  7.  9. 10. 11.]
 [ 2.  3.  4.  6.  7.  8. 10. 11. 12.]
 [ 3.  4.  0.  7.  8.  0. 11. 12.  0.]
 [ 0.  5.  6.  0.  9. 10.  0. 13. 14.]
 [ 5.  6.  7.  9. 10. 11. 13. 14. 15.]
 [ 6.  7.  8. 10. 11. 12. 14. 15. 16.]
 [ 7.  8.  0. 11. 12.  0. 15. 16.  0.]
 [ 0.  9. 10.  0. 13. 14.  0.  0.  0.]
 [ 9. 10. 11. 13. 14. 15.  0.  0.  0.]
 [10. 11. 12. 14. 15. 16.  0.  0.  0.]
 [11. 12.  0. 15. 16.  0.  0.  0.  0.]]
[[[[ 1.  2.  3.  4.]
   [ 5.  6.  7.  8.]
   [ 9. 10. 11. 12.]
   [13. 14. 15. 16.]]]]
```

#### Define linear transformation

In [4]:
def linear_forward(w, x, b):
    """Compute y = w @ x^T + b, transposed back to (samples, features)."""
    y = np.matmul(w, x.T).T + b
    return y

#### Verifying im2col functionality

In [5]:
np.random.seed(1)

x_imag = np.random.randn(1,4,4,2)
x_wegt = np.random.randn(4,3,3,2)
x_bias = np.random.randn(4)

x_b, x_h, x_w, x_c = x_imag.shape
f_c, f_h, f_w, f_i = x_wegt.shape

c_imag = im2col(x_imag, 3, 3, 1, 1, 1, 1) # (xb,xh,xw,xc) -> (xb*xh*xw,fh*fw*xc)
c_wegt = x_wegt.reshape(f_c, -1)          # (fc,fh,fw,fi) -> (fc,fh*fw*fi) where{fi==xc}

c_fwrd = linear_forward(c_wegt, c_imag, x_bias)
b_imag = c_fwrd.reshape(x_b, x_h, x_w, f_c)

print(x_imag.shape, x_wegt.shape, x_bias.shape)
print(c_imag.shape, c_wegt.shape, c_fwrd.shape, b_imag.shape)
print(b_imag[0,:,:,0])

(1, 4, 4, 2) (4, 3, 3, 2) (4,)
(16, 18) (4, 18) (16, 4) (1, 4, 4, 4)
[[-3.8435009  -6.48930166 -3.50702806 -2.89073585]
 [-7.76972217  1.00800395 -1.66909802 -0.52482669]
 [-3.51166849 -1.704174   -0.46271359 -4.20899685]
 [-1.93644556  4.16773899 -3.25422526 -1.4839155 ]]


**Expected Output:**
```
(1, 4, 4, 2) (4, 3, 3, 2) (4,)
(16, 18) (4, 18) (16, 4) (1, 4, 4, 4)
[[-3.8435009  -6.48930166 -3.50702806 -2.89073585]
 [-7.76972217  1.00800395 -1.66909802 -0.52482669]
 [-3.51166849 -1.704174   -0.46271359 -4.20899685]
 [-1.93644556  4.16773899 -3.25422526 -1.4839155 ]]
```

### Pooling functions: MaxPool, AvgPool, and GlobalPool

In [ ]:
def pooling(x_img, flt_h, flt_w, stride_h=1, stride_w=1, filter_type='max', padding=False):
    x_b, x_h, x_w, x_c = x_img.shape
    padding_h, padding_w = (flt_h//2, flt_w//2) if padding else (0, 0)

    ### START CODE HERE ###

    # Transpose and reshape to apply im2col per channel independently
    t_img = None                # (b, c, h, w)
    r_img = None                # merge batch and channel
    # Rearrange into column matrix for pooling windows
    x_col = None
    # Compute output dimensions (Eq. 5.9)
    n_h = None
    n_w = None

    if filter_type=='max':
        # Max pooling: select maximum in each window (Section 5.4, Eq. 5.13)
        pmask = None
        pmask[np.arange(x_col.shape[0]), np.argmax(x_col, axis=-1)] = 1
        pmask = None
        pmask = pmask.reshape(t_img.shape).transpose(0,2,3,1)
        x_new = None
        x_new = x_new.reshape(x_b, x_c, n_h, n_w).transpose(0,2,3,1)
    elif filter_type=='average':
        # Average pooling: compute mean over each window (Section 5.4, Eq. 5.11)
        pmask = None
        x_new = None
        x_new = x_new.reshape(x_b, x_c, n_h, n_w).transpose(0,2,3,1)
    elif filter_type=='global':
        # Global average pooling: mean over entire spatial extent (Section 5.4)
        pmask = None
        f_img = t_img.reshape(-1, x_h*x_w)
        x_new = None
        x_new = x_new.reshape(x_b, -1)

    ### END CODE HERE ###
    
    else:
        print('pooling type error')

    return x_new, pmask

In [ ]:
class myPoolingLayer:
    def __init__(self, flt_h, flt_w, stride_h, stride_w, p_type='max'):
        self.type = p_type
        self.f_h = flt_h
        self.f_w = flt_w
        self.s_h = stride_h
        self.s_w = stride_w
        self.mask = None

    def forward(self, x):
        x, m = pooling(x, self.f_h, self.f_w, self.s_h, self.s_w, self.type, padding=False)
        self.mask = m
        return x

    def backward(self, x):
        """Backpropagation for pooling (Section 5.4, Eqs. 5.12 and 5.14).
        Note: this implementation assumes non-overlapping pooling (stride == filter size)."""
        (img_b, img_h, img_w, img_c) = self.mask.shape

        ### START CODE HERE ###

        if self.type=='max':
            # Max pooling backward: route gradient only to max positions (Eq. 5.14)
            x_eh = None
            x_ex = None
            x_gr = None             # zero out non-max positions
        elif self.type=='average':
            # Average pooling backward: distribute gradient uniformly (Eq. 5.12)
            x_eh = None
            x_ex = None
            x_gr = None             # divide by N = f_h * f_w
        elif self.type=='global':
            x_b, x_c = x.shape
            # Global pooling backward: distribute gradient over all spatial positions
            x_ed = None
            x_eh = None
            x_ex = None
            x_gr = None             # divide by total spatial size

        ### END CODE HERE ###
        
        else:
            print('pooling type error in backward')
        return x_gr

#### Test forward max pooling

In [8]:
plyr = myPoolingLayer(flt_h=2, flt_w=2, stride_h=2, stride_w=2, p_type='max')
# Test code here
np.random.seed(1)

x_imag = np.random.randint(1,100,(2,6,6,3))
p_imag = plyr.forward(x_imag)

print('x:', x_imag.shape, 'p:', p_imag.shape, 'm:', plyr.mask.shape)
print(x_imag[0,:,:,0])
print(p_imag[0,:,:,0])
print(plyr.mask[0,:,:,0])

x: (2, 6, 6, 3) p: (2, 3, 3, 3) m: (2, 6, 6, 3)
[[38 10 80  2  7 21]
 [12 15 88 97 10 62]
 [ 2 82 14 31 71 58]
 [25 27 42 65 99 27]
 [10 28 84 33 24 26]
 [75 33 56  4  7 71]]
[[38. 97. 62.]
 [82. 65. 99.]
 [75. 84. 71.]]
[[1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 1.]]


**Expected Output**:
```
x: (2, 6, 6, 3) p: (2, 3, 3, 3) m: (2, 6, 6, 3)
[[38 10 80  2  7 21]
 [12 15 88 97 10 62]
 [ 2 82 14 31 71 58]
 [25 27 42 65 99 27]
 [10 28 84 33 24 26]
 [75 33 56  4  7 71]]
[[38. 97. 62.]
 [82. 65. 99.]
 [75. 84. 71.]]
[[1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 1.]
 [0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 1. 1. 0.]
 [0. 0. 1. 0. 0. 0.]
 [1. 0. 0. 0. 0. 1.]]
```

#### Test backward max pooling

In [9]:
plyr = myPoolingLayer(flt_h=2, flt_w=2, stride_h=2, stride_w=2, p_type='max')

np.random.seed(1)

x_imag = np.random.randint(1,100,(2,6,6,3))
p_imag = plyr.forward(x_imag)

x_grad = plyr.backward(p_imag)

print(x_imag[1,:,:,0])
print(x_grad[1,:,:,0])

[[92  8 76 21  8 58]
 [14 82 75 33 95 83]
 [93 55 87 72 16 43]
 [23 54 97 57 97 15]
 [44 57 25 72 37 78]
 [48 79 17 68 47 64]]
[[92.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0. 95.  0.]
 [ 0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0. 78.]
 [ 0. 79.  0.  0.  0.  0.]]


**Expected Output**:
```
[[92  8 76 21  8 58]
 [14 82 75 33 95 83]
 [93 55 87 72 16 43]
 [23 54 97 57 97 15]
 [44 57 25 72 37 78]
 [48 79 17 68 47 64]]
[[92.  0. 76.  0.  0.  0.]
 [ 0.  0.  0.  0. 95.  0.]
 [93.  0.  0.  0.  0.  0.]
 [ 0.  0. 97.  0. 97.  0.]
 [ 0.  0.  0. 72.  0. 78.]
 [ 0. 79.  0.  0.  0.  0.]]
```

#### Verifying pooling with various filter and stride sizes

In [10]:
# Case 1: stride of 1  pooling(x_img, flt_h, flt_w, stride_h=1, stride_w=1, filter_type='max', padding=False)
np.random.seed(1)
x_imag = np.random.randn(2, 5, 5, 3)

p_imag, mask = pooling(x_imag, 3, 3, 1, 1, 'max')
print("mode = max")
print("p.shape = " + str(p_imag.shape))
print("p[0][0:2] =\n", p_imag[0,:2])
print()
p_imag, mask = pooling(x_imag, 3, 3, 1, 1, 'average')
print("mode = average")
print("p.shape = " + str(p_imag.shape))
print("p[0][0:2] =\n", p_imag[0,:2])

mode = max
p.shape = (2, 3, 3, 3)
p[0][0:2] =
 [[[1.74481176 0.90159072 1.65980218]
  [1.74481176 1.46210794 1.65980218]
  [1.74481176 1.6924546  1.65980218]]

 [[1.14472371 0.90159072 2.10025514]
  [1.14472371 0.90159072 1.65980218]
  [1.14472371 1.6924546  1.65980218]]]

mode = average
p.shape = (2, 3, 3, 3)
p[0][0:2] =
 [[[-0.03010467 -0.00324021 -0.33629886]
  [ 0.14331048  0.19314675 -0.4449052 ]
  [ 0.12893444  0.22242847  0.1250676 ]]

 [[-0.3818019   0.01599935  0.17056271]
  [ 0.04737072  0.02592447  0.09203384]
  [ 0.03970486  0.15718909  0.34530249]]]


**Expected Output**:
```
mode = max
p.shape = (2, 3, 3, 3)
p[0][0:2] =
 [[[1.74481176 0.90159072 1.65980218]
  [1.74481176 1.46210794 1.65980218]
  [1.74481176 1.6924546  1.65980218]]

 [[1.14472371 0.90159072 2.10025514]
  [1.14472371 0.90159072 1.65980218]
  [1.14472371 1.6924546  1.65980218]]]

mode = average
p.shape = (2, 3, 3, 3)
p[0][0:2] =
 [[[-0.03010467 -0.00324021 -0.33629886]
  [ 0.14331048  0.19314675 -0.4449052 ]
  [ 0.12893444  0.22242847  0.1250676 ]]

 [[-0.3818019   0.01599935  0.17056271]
  [ 0.04737072  0.02592447  0.09203384]
  [ 0.03970486  0.15718909  0.34530249]]]
```

In [11]:
# Case 2: stride of 2
np.random.seed(1)
x_imag = np.random.randn(2, 5, 5, 3)

p_imag, mask = pooling(x_imag, 3, 3, 2, 2, 'max')
print("mode = max")
print("p.shape = " + str(p_imag.shape))
print("p[0] =\n", p_imag[0])
print()

p_imag, mask = pooling(x_imag, 3, 3, 2, 2, 'average')
print("mode = average")
print("p.shape = " + str(p_imag.shape))
print("p[0] =\n", p_imag[0])

mode = max
p.shape = (2, 2, 2, 3)
p[0] =
 [[[1.74481176 0.90159072 1.65980218]
  [1.74481176 1.6924546  1.65980218]]

 [[1.13162939 1.51981682 2.18557541]
  [1.13162939 1.6924546  2.18557541]]]

mode = average
p.shape = (2, 2, 2, 3)
p[0] =
 [[[-0.03010467 -0.00324021 -0.33629886]
  [ 0.12893444  0.22242847  0.1250676 ]]

 [[-0.38268052  0.23257995  0.6259979 ]
  [-0.09525515  0.268511    0.46605637]]]


**Expected Output**:
```
mode = max
p.shape = (2, 2, 2, 3)
p[0] =
 [[[1.74481176 0.90159072 1.65980218]
  [1.74481176 1.6924546  1.65980218]]

 [[1.13162939 1.51981682 2.18557541]
  [1.13162939 1.6924546  2.18557541]]]

mode = average
p.shape = (2, 2, 2, 3)
p[0] =
 [[[-0.03010467 -0.00324021 -0.33629886]
  [ 0.12893444  0.22242847  0.1250676 ]]

 [[-0.38268052  0.23257995  0.6259979 ]
  [-0.09525515  0.268511    0.46605637]]]
```

#### Test forward global pooling

In [12]:
plyr = myPoolingLayer(flt_h=2, flt_w=2, stride_h=2, stride_w=2, p_type='global')

np.random.seed(1)

x_imag = np.random.randn(2,6,6,3)
g_imag = plyr.forward(x_imag)

print(g_imag.shape)
print(g_imag[1], '<=', np.mean(x_imag[1,:,:,0]))

(2, 3)
[ 0.16776591 -0.0540342   0.16849378] <= 0.16776591385427878


**Expected Output**:
```
(2, 3)
[ 0.16776591 -0.0540342   0.16849378] <= 0.16776591385427878
```

#### Test backward global pooling

In [13]:
plyr = myPoolingLayer(flt_h=2, flt_w=2, stride_h=2, stride_w=2, p_type='global')

np.random.seed(1)

x_imag = np.random.rand(2,6,6,3)
g_imag = plyr.forward(x_imag)

x_grad = plyr.backward(g_imag)

print(x_grad.shape, g_imag[1]/36)
print(x_grad[1,:,:,0])

(2, 6, 6, 3) [0.01230655 0.01500427 0.01369827]
[[0.01230655 0.01230655 0.01230655 0.01230655 0.01230655 0.01230655]
 [0.01230655 0.01230655 0.01230655 0.01230655 0.01230655 0.01230655]
 [0.01230655 0.01230655 0.01230655 0.01230655 0.01230655 0.01230655]
 [0.01230655 0.01230655 0.01230655 0.01230655 0.01230655 0.01230655]
 [0.01230655 0.01230655 0.01230655 0.01230655 0.01230655 0.01230655]
 [0.01230655 0.01230655 0.01230655 0.01230655 0.01230655 0.01230655]]


**Expected Output**:
```
(2, 6, 6, 3) [0.01230655 0.01500427 0.01369827]
[[0.01230655 0.01230655 0.01230655 0.01230655 0.01230655 0.01230655]
 [0.01230655 0.01230655 0.01230655 0.01230655 0.01230655 0.01230655]
 [0.01230655 0.01230655 0.01230655 0.01230655 0.01230655 0.01230655]
 [0.01230655 0.01230655 0.01230655 0.01230655 0.01230655 0.01230655]
 [0.01230655 0.01230655 0.01230655 0.01230655 0.01230655 0.01230655]
 [0.01230655 0.01230655 0.01230655 0.01230655 0.01230655 0.01230655]]
```

### 2D convolution layer

This class implements forward propagation (Eq. 5.4) and backward propagation
(Eqs. 5.5–5.8) for a 2D convolutional layer using the im2col method (Section 5.7).

In [ ]:
class myConv2DLayer:
    def __init__(self, n_out, flt_h, flt_w, n_in, stride=1, padding=True):
        self.weight = np.zeros((n_out, flt_h, flt_w, n_in))
        self.bias = np.zeros((n_out,))
        self.f_h = flt_h
        self.f_w = flt_w
        self.f_c = n_out          # C_O: number of output channels
        self.f_i = n_in           # C_I: number of input channels
        self.s_h = stride
        self.s_w = stride
        self.p_h = flt_h // 2 if padding else 0
        self.p_w = flt_w // 2 if padding else 0
        self.saved_x = None       # store input for backpropagation

    def forward(self, x):
        """Forward propagation: z = conv(x, w) + b (Eq. 5.4)"""
        x_b, x_h, x_w, _ = x.shape

        ### START CODE HERE ###

        # Save input for use in backward pass
        self.saved_x = None

        # Compute output spatial dimensions (Eq. 5.9)
        x_h = None
        x_w = None

        # Rearrange input patches into columns (Section 5.7, im2col)
        c_img = None
        # Reshape filter to 2D: (C_O, F_H*F_W*C_I)
        c_wgt = None
        # Matrix multiply and add bias: cross-correlation as matmul (Eq. 5.4)
        c_lin = None
        # Reshape result to 4D output tensor: (b, n_h, n_w, C_O)
        x_lin = None

        ### END CODE HERE ###

        return x_lin

    def backward(self, x):
        """Backward propagation for convolution (Eqs. 5.5-5.8).
        Args:
            x: upstream gradient dJ/dz, shape (b, n_h, n_w, C_O)
        Returns:
            dw: weight gradient (Eq. 5.6)
            db: bias gradient (Eq. 5.5)
            wdJdz: input gradient (Eq. 5.8)
        """
        x_b, x_h, x_w, x_c = self.saved_x.shape

        ### START CODE HERE ###

        # Prepare dilated gradient tensor for stride > 1
        st_h = None
        st_w = None
        x_gr = None             # start with zeros and insert upstream gradients at strided positions
        # Insert upstream gradients at strided positions (Section 5.3, Fig. 5.10)
        x_gr[:, st_h::self.s_h, st_w::self.s_w, :] = x

        # Weight gradient: cross-correlation of input with dJ/dz (Eq. 5.6)
        xi_trn = None           # (C_I, h, w, b)
        c_tran = None           # (C_O, h, w, b) rearrange input for matrix multiplication
        x_tran = None           # (C_O, h, w, b)
        c_x_dz = None           # (C_O, h*w*b)
        w_grad = None
        dw = None

        # Bias gradient: sum over all spatial positions and batch (Eq. 5.5)
        b_grad = None
        db = None

        # Input gradient: convolution of dJ/dz with flipped kernel (Eq. 5.8)
        c_dJdz = None
        # Flip kernel along both spatial axes (Eq. 5.7: reversed kernel indices)
        f_wegt = None
        c_wegt = None
        w_dJda = None
        wdJdz = None

        ### END CODE HERE ###

        return dw, db, wdJdz

#### Test Conv2D layer

In [15]:
lyr = myConv2DLayer(n_out=1, flt_h=3, flt_w=3, n_in=1)

lyr.weight = np.array(
    [   [1,0,-1],
        [2,0,-2],
        [1,0,-1]  ]).reshape(1,3,3,1)
X = np.array(
    [   [1,1,1,2,3],
        [1,1,1,2,3],
        [1,1,1,2,3],
        [2,2,2,2,3],
        [3,3,3,3,3],
        [4,4,4,4,4]  ]).reshape(1,6,5,1)

y = lyr.forward(X)
y[0,:,:,0]

array([[ -3.,   0.,  -3.,  -6.,   6.],
       [ -4.,   0.,  -4.,  -8.,   8.],
       [ -5.,   0.,  -3.,  -7.,   8.],
       [ -8.,   0.,  -1.,  -4.,   9.],
       [-12.,   0.,   0.,  -1.,  12.],
       [-11.,   0.,   0.,   0.,  11.]])

**Expected Output**:
```
array([[ -3.,   0.,  -3.,  -6.,   6.],
       [ -4.,   0.,  -4.,  -8.,   8.],
       [ -5.,   0.,  -3.,  -7.,   8.],
       [ -8.,   0.,  -1.,  -4.,   9.],
       [-12.,   0.,   0.,  -1.,  12.],
       [-11.,   0.,   0.,   0.,  11.]])
```

In [16]:
lyr = myConv2DLayer(n_out=1, flt_h=3, flt_w=3, n_in=1)

lyr.weight = np.array(
    [   [1,0,-1],
        [2,0,-2],
        [1,0,-1]  ]).reshape(1,3,3,1)
X = np.array(
    [   [1,1,1,2,3],
        [1,1,1,2,3],
        [1,1,1,2,3],
        [2,2,2,2,3],
        [3,3,3,3,3],
        [4,4,4,4,4]  ]).reshape(1,6,5,1)
d = np.array(
    [   [0,0,0,0,0],
        [0,1,1,1,0],
        [0,1,1,1,0],
        [0,1,1,1,0],
        [0,1,1,1,0],
        [0,0,0,0,0] ]).reshape(1,6,5,1)

a = lyr.forward(X)
dw, db, z = lyr.backward(d)

print(dw[0,:,:,0])
print(db[0])
print(z[0,:,:,0])

[[15. 18. 25.]
 [21. 23. 28.]
 [30. 31. 34.]]
12.0
[[ 1.  1.  0. -1. -1.]
 [ 3.  3.  0. -3. -3.]
 [ 4.  4.  0. -4. -4.]
 [ 4.  4.  0. -4. -4.]
 [ 3.  3.  0. -3. -3.]
 [ 1.  1.  0. -1. -1.]]


**Expected Output**:
```
[[15. 18. 25.]
 [21. 23. 28.]
 [30. 31. 34.]]
12.0
[[ 1.  1.  0. -1. -1.]
 [ 3.  3.  0. -3. -3.]
 [ 4.  4.  0. -4. -4.]
 [ 4.  4.  0. -4. -4.]
 [ 3.  3.  0. -3. -3.]
 [ 1.  1.  0. -1. -1.]]
```

#### Verifying Conv2D layer

In [17]:
lyr = myConv2DLayer(n_out=8, flt_h=3, flt_w=3, n_in=4, stride=2, padding=True)

np.random.seed(1)
A_prev = np.random.randn(10,5,7,4)
W = np.random.randn(8,3,3,4)
b = np.random.randn(8)

lyr.weight = W
lyr.bias = b

Z = lyr.forward(A_prev)
# cache_conv = conv_forward(A_prev, W, b, hparameters)
print("Z's mean =\n", np.mean(Z))
print("Z[3,2,1] =\n", Z[3,2,1])

Z's mean =
 0.29939433759873685
Z[3,2,1] =
 [-5.57294243  1.88740034 11.31730088 -0.0464875  -2.98511023 -4.39348933
 -1.75801838  4.60767283]


**Expected Output**:
```
Z's mean =
 0.29939433759873696
Z[3,2,1] =
 [-5.57294243  1.88740034 11.31730088 -0.0464875  -2.98511023 -4.39348933
 -1.75801838  4.60767283]
```

In [18]:
# We'll run conv_forward to initialize the 'Z' and 'cache_conv",
# which we'll use to test the conv_backward function
lyr = myConv2DLayer(n_out=8, flt_h=3, flt_w=3, n_in=4, stride=2, padding=True)

np.random.seed(1)
A_prev = np.random.randn(10,5,5,4)
W = np.random.randn(8,3,3,4)
b = np.random.randn(8)

lyr.weight = W
lyr.bias = b

Z = lyr.forward(A_prev)

print(Z.shape)

# Test conv_backward
dW, db, dA = lyr.backward(Z)

print("dA_mean =", np.mean(dA))
print("dW_mean =", np.mean(dW))
print("db_mean =", np.mean(db))

assert dA.shape == (10, 5, 5, 4), f"Wrong shape for dA  {dA.shape} != (10, 5, 5, 4)"
assert dW.shape == (8, 3, 3, 4), f"Wrong shape for dW {dW.shape} != (8, 3, 3, 4)"
assert db.shape == (8,), f"Wrong shape for db {db.shape} != (8,)"
assert np.isclose(np.mean(dA), 1.2728759), "Wrong values for dA"
assert np.isclose(np.mean(dW), -0.0961898), "Wrong values for dW"
assert np.isclose(np.mean(db), 0.05888198), "Wrong values for db"

print("\033[92m All tests passed.")

(10, 3, 3, 8)
dA_mean = 1.2728759025598175
dW_mean = -0.09618986791877772
db_mean = 0.0588819877676956
 All tests passed.


**Expected Output:**
```
dA_mean = 1.2728759025598175
dW_mean = -0.09618986791877772
db_mean = 0.0588819877676956
```

(c) 2026 S. W. Lee